### Latin square Simulation

- Latin square is the square with all rows and columns has distinct symbol: https://en.wikipedia.org/wiki/Latin_square
- In this notebook, the author will simulate the latent square for reasonable dimensions <= 11 and provide an est for total number of latent squares using SMC
- Additionally, the same idea will be applied to solve a sudoku puzzle. 

Let's start with some setup! Define:  
- $\pi_0$ is the uniform distribution of all squares n x n with each row is already a permutation from $1 - n$
so that we only need to consider the validity of columns.

- $\pi_1$ is the uniform distribution of all Latin squares n x n.

We will start from $\pi_0$ to generate samples from $\pi_T$

- Let's say if we directly apply the ideas of SMCS by building a list of intermediate distributions $\pi_0^{(1 - \lambda)} * \pi_1^{\lambda}$, what happen is that for all lambdas > 0, all intermediate dists will be collapsed to the target distribution as its pdf is exactly the same aft normalizing (i.e. = 0 if the square is non-Latin and = 1 otherwise)

The idea is interesting as we will try to not directly sample from our target dist but from an "approximate" version of it. 
Let's consider: 

$target = e^{-lambda * score(X)}$ with lambda is big enough value let say 1000. 

- Score(X) here is a score function to measure how "bad" a sample from a Latin properties.
- When X is Latin square, score(X) will be zero and target = 1, otherwise, this value is very small, thanks to the big lambda. By doing SMC on this alternative dist, most of the samples we received will expect to be Latin.

In [1]:
from libs import MCMC, SMC, AcceptanceTracker, test_samples

First of all, define a scorer class to maintain all scorer methods we will have 

In [2]:
from jax import numpy as jnp

import numpy as np 

class Scorer:
    @staticmethod
    def score_repetitive_columns(flat_board: jnp.array):
        n = np.sqrt(len(flat_board)).astype(int)
        assert len(flat_board) == n * n
        
        score = 0 
        for c in range(n):
            seen = jnp.array([False for _ in range(10)])
            for x in flat_board[c::n]:
                score += jnp.where(seen[x], 1, 0)
                seen = seen.at[x].set(True)
        return score 

    @staticmethod
    def score_sudoku(flat_board: np.array):
        #TODO: implement
        return 0 

assert Scorer.score_repetitive_columns(jnp.array([1,2,1,2])) == 2
assert Scorer.score_repetitive_columns(jnp.array([1,2,2,1])) == 0

#### Latin Square Solver

Secondly, define the prior/target dist logpdfs we will sample from as well as the proposed_fn to move to the next state


In [3]:
import jax
import jax.numpy as jnp
from jax import random

import numpy as np 

def prior_dist_logpdf(flat_board: jnp.array):
    """
    Supports:
        flat_board.shape == (N,)
        flat_board.shape == (B, N)
    """
    board_size = flat_board.shape[-1]
    n = int(np.sqrt(board_size))

    log_prob = -n * jnp.log(jnp.arange(1, n + 1)).sum()

    if flat_board.ndim == 1:
        return log_prob
    else:
        return jnp.full((flat_board.shape[0],), log_prob)

def target_dist_logpdf(flat_board: jnp.array, score_fn, lam=1000):
    """
    Supports:
        (N,)  -> scalar
        (B,N) -> (B,)
    """
    if flat_board.ndim == 1:
        return -lam * score_fn(flat_board)
    else:
        scores = jax.vmap(score_fn)(flat_board)
        return -lam * scores

In [4]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np

def proposed_fn(flat_board: jnp.array, key):
    """
    Supports:
        (N,)
        (B,N)
    """

    def propose_one(board: np.array, key):
        board_size = board.shape[0]
        n = jnp.sqrt(board_size).astype(int)

        key1, key2 = random.split(key)

        row = random.randint(key1, (), 0, n)
        col = random.randint(key2, (), 0, n - 1)

        i = row * n + col
        j = i + 1

        board_i, board_j = board[i], board[j]
        board = board.at[i].set(board_j)
        board = board.at[j].set(board_i)
        return board 

    if flat_board.ndim == 1:
        return propose_one(flat_board, key)

    else:
        keys = random.split(key, flat_board.shape[0])
        return jax.vmap(propose_one)(flat_board, keys)

In [16]:
from jax import random, vmap
from jax import numpy as jnp 
from functools import partial

class LatinSquareSampler: 
    def __init__(self, n: int, score_fn):
        self.__score_fn = score_fn
        self.__key = random.key(100)
        self.__n = n  
        self.smc = SMC(
            dims = n * n, 
            target_dist_logpdf = partial(
                target_dist_logpdf, 
                score_fn = Scorer.score_repetitive_columns
            ), 
            prior_dist_logpdf = prior_dist_logpdf, 
            proposed_fn = proposed_fn, 
            key = self._split_key()[0]
        )
        self.log_z = n * jnp.log(jnp.arange(1, n + 1)).sum()

    def _split_key(self, n = 2):
        split_keys = random.split(self.__key, n)
        self.__keys = split_keys[0]
        return split_keys[1:]

    def __generate_initial_state(self, key):
        sub_keys = random.split(key, num = self.__n)
        perm = vmap(
            lambda key: random.permutation(key, jnp.arange(1, self.__n + 1)) 
        )(sub_keys)
        return perm.reshape(-1)

    def __generate_initial_states(self, no_samples: int):
        sub_keys = self._split_key(no_samples + 1)
        states = vmap(
            lambda key: self.__generate_initial_state(key)
        )(sub_keys)
        return states
    
    def run_smc(self, no_samples: int):
        samples = self.__generate_initial_states(no_samples)
        assert samples.shape == (no_samples, self.__n * self.__n)
        
        self.smc.reset(samples = samples)
        lam_list, diff_log_z = self.smc.build_intermediate_dists(max_steps=64, n_bisect=30, mcmc_iters=50)
        print(lam_list, diff_log_z)
        self.log_z += diff_log_z
        return lam_list

    def get_current_sample_list(self):
        return self.smc.get_current_sample_list()

    def est_total(self):
        return jnp.exp(self.log_z)  

Now let's test our solver!

In [17]:
grouth_truth = [
    1, 
    2, 
    12, 
    576, 
    161280, 
    812851200, 
    61479419904000, 
    108776032459082956800, 
    5524751496156892842531225600, 
    9982437658213039871725064756920320000, 
    776966836171770144107444346734230682311065600000
]

In [ ]:
NDIMS = 10 
latin_solver = LatinSquareSampler(
    n = NDIMS, 
    score_fn = Scorer.score_repetitive_columns
)

lam_list = latin_solver.run_smc(
    no_samples = 500_000
)
samples = latin_solver.get_current_sample_list()
for sample in samples[:10]:
    print(Scorer.score_repetitive_columns(sample))

Current loop value: 0.0
Current loop value: 0.00016343127936124802
Current loop value: 0.00032813765574246645
Current loop value: 0.0004950700094923377
Current loop value: 0.0006638305494561791
Current loop value: 0.0008351155556738377
Current loop value: 0.0010093527380377054
Current loop value: 0.0011868642177432775
Current loop value: 0.0013676974922418594
Current loop value: 0.0015534234698861837
Current loop value: 0.0017434186302125454
Current loop value: 0.0019381616730242968
Current loop value: 0.0021385657601058483
Current loop value: 0.00234454357996583
Current loop value: 0.0025556939654052258
Current loop value: 0.002769947052001953
Current loop value: 0.0029864036478102207
Current loop value: 0.003207341767847538
Current loop value: 0.003441842272877693
Current loop value: 0.0037101181223988533


Display some Latin square samples 

In [ ]:
for sample in samples[:10]:
    print(sample.reshape(NDIMS, NDIMS), "\n")

In [14]:
latin_solver.est_total()

Array(inf, dtype=float32)

#### Sudoku Solver

In [ ]:
from jax import random
from functools import partial

class SudokuSolver: 
    def __init__(self, n: int, configuration: jnp.array, score_fn):
        self.__score_fn = score_fn
        self.smc = SMC(
            dims = n * n, 
            target_dist_logpdf = target_dist_logpdf, 
            prior_dist_logpdf = prior_dist_logpdf, 
            proposed_fn = proposed_fn, 
            key = random.key(0)
        )
        self.z = 0 
        self.configuration = configuration

    def run_smc(self):
        r"""
            build intermediate + samples from the valid sudoku   
        """
        ... 

    def solve(self):
        r"""
            iteratively take from smc sample until has something with score = 0 
        """
        ... 